# Backend-11 OEE (Notebook)
이 노트북은 **2026-02-02_추가_백엔드-11_설계.pdf** + 사용자가 확정한 규칙을 그대로 반영하여,
- station(Vision1/Vision2) × remark(PD/Non-PD) 별
- **Ideal 총생산 / 실제(PASS) 총생산** 기반 OEE를 계산합니다.

확정 규칙 반영 요약:
- Shift window 길이: **42000초** (end 포함, duration은 `end-start`로 계산)
- remark change `at_time`: **이전 remark의 마지막 시각(포함)**, 다음 remark는 `at_time+1s`부터
- planned stop: **구간 겹침은 합집합 길이**, remark 구간과 **overlap**해서 remark별 planned stop 산정
- remark change 이벤트가 없으면: planned stop은 **total_planned_time**(또는 union 계산) 사용 + remark는 `a_station_*_daily_final_amount`의 단일 remark 사용
- Ideal_ct는 **20260128 day/night 기준**으로 고정


In [1]:
# =========================
# 0) 설치(필요 시)
# =========================
# !pip -q install sqlalchemy psycopg2-binary pandas python-dateutil

from __future__ import annotations

import os
import re
from dataclasses import dataclass
from datetime import datetime, date, time, timedelta
from zoneinfo import ZoneInfo
from typing import List, Tuple, Dict, Optional

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

KST = ZoneInfo("Asia/Seoul")


In [2]:
# =========================
# 1) 파라미터
# =========================
PROD_DAY = "20260128"   # YYYYMMDD
SHIFT_TYPE = "day"      # "day" | "night"

# shift window: end 포함 / duration은 end-start => 43199초 고정
WINDOW_SECONDS = 43199

STATIONS = ["Vision1", "Vision2"]
REMARKS = ["PD", "Non-PD"]

# Ideal_ct 기준일(고정)
IDEAL_BASE_PROD_DAY = "20260128"

# =========================
# 2) DB 설정
# =========================
# (권장) 환경변수로 override 가능:
#   PGHOST, PGPORT, PGDATABASE, PGUSER, PGPASSWORD
DB_CONFIG = {
    "host": os.getenv("PGHOST", "100.105.75.47"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE", "postgres"),
    "user": os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", ""),
}

def make_engine() -> Engine:
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    return create_engine(
        url,
        pool_size=1,
        max_overflow=0,
        pool_pre_ping=True,
        pool_recycle=1800,
    )

engine = make_engine()


In [3]:
# =========================
# 3) 유틸: 시간/윈도우/구간(합집합/겹침)
# =========================
def yyyymmdd_to_date(s: str) -> date:
    return date(int(s[:4]), int(s[4:6]), int(s[6:8]))

def parse_hms(s: str) -> time:
    # "HH:MI:SS" 전제
    hh, mm, ss = s.strip().split(":")
    return time(int(hh), int(mm), int(ss))

def build_shift_window(prod_day: str, shift_type: str) -> tuple[datetime, datetime]:
    d = yyyymmdd_to_date(prod_day)
    if shift_type == "day":
        start = datetime.combine(d, time(8, 30, 0), tzinfo=KST)
        # end 포함이지만 duration은 end-start=43199
        end = datetime.combine(d, time(20, 29, 59), tzinfo=KST)
    elif shift_type == "night":
        start = datetime.combine(d, time(20, 30, 0), tzinfo=KST)
        end = datetime.combine(d + timedelta(days=1), time(8, 29, 59), tzinfo=KST)
    else:
        raise ValueError("SHIFT_TYPE must be 'day' or 'night'")
    # 안전 체크: 43199초
    dur = int((end - start).total_seconds())
    if dur != WINDOW_SECONDS:
        raise ValueError(f"Window seconds mismatch: got {dur}, expected {WINDOW_SECONDS}")
    return start, end

@dataclass(frozen=True)
class Interval:
    start: datetime
    end: datetime  # inclusive end semantic, but duration uses end-start seconds (per spec)

def merge_intervals(intervals: List[Interval]) -> List[Interval]:
    """겹치는/맞닿는 구간을 합집합으로 병합. (end 포함 semantics)"""
    if not intervals:
        return []
    intervals = sorted(intervals, key=lambda x: x.start)
    merged = [intervals[0]]
    for cur in intervals[1:]:
        last = merged[-1]
        # 맞닿는/겹치는 조건: cur.start <= last.end (+1초?) 
        # 여기서는 end 포함이지만 duration을 end-start로 쓰므로,
        # [A,B]와 [B,B2]는 실제로 0초 gap으로 연속으로 취급해도 OK.
        if cur.start <= last.end:
            merged[-1] = Interval(last.start, max(last.end, cur.end))
        else:
            merged.append(cur)
    return merged

def overlap_seconds(a: Interval, b: Interval) -> int:
    """두 구간의 겹치는 길이(초). 규칙: duration = end-start (end 포함일 때도 +1 하지 않음)"""
    s = max(a.start, b.start)
    e = min(a.end, b.end)
    if e <= s:
        return 0
    return int((e - s).total_seconds())

def total_seconds(intervals: List[Interval]) -> int:
    return sum(int((iv.end - iv.start).total_seconds()) for iv in intervals)

WINDOW_START, WINDOW_END = build_shift_window(PROD_DAY, SHIFT_TYPE)
WINDOW_START, WINDOW_END


(datetime.datetime(2026, 1, 28, 8, 30, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')),
 datetime.datetime(2026, 1, 28, 20, 29, 59, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')))

In [4]:
# =========================
# 4) 테이블 선택
# =========================
SAVE_SCHEMA = "i_daily_report"

T_REMARK_CHANGE_DAY   = 'j_remark_change_day_daily'
T_REMARK_CHANGE_NIGHT = 'j_remark_change_night_daily'

T_PLANNED_STOP_DAY    = 'i_planned_stop_time_day_daily'
T_PLANNED_STOP_NIGHT  = 'i_planned_stop_time_night_daily'

T_FINAL_AMT_DAY       = 'a_station_day_daily_final_amount'
T_FINAL_AMT_NIGHT     = 'a_station_night_daily_final_amount'

IDEAL_SCHEMA = "e1_FCT_ct"
IDEAL_TABLE  = "fct_whole_op_ct"

remark_change_table = T_REMARK_CHANGE_DAY if SHIFT_TYPE == "day" else T_REMARK_CHANGE_NIGHT
planned_stop_table  = T_PLANNED_STOP_DAY  if SHIFT_TYPE == "day" else T_PLANNED_STOP_NIGHT
final_amt_table     = T_FINAL_AMT_DAY     if SHIFT_TYPE == "day" else T_FINAL_AMT_NIGHT

remark_change_fqn = f'"{SAVE_SCHEMA}"."{remark_change_table}"'
planned_stop_fqn  = f'"{SAVE_SCHEMA}"."{planned_stop_table}"'
final_amt_fqn     = f'"{SAVE_SCHEMA}"."{final_amt_table}"'
ideal_fqn         = f'"{IDEAL_SCHEMA}"."{IDEAL_TABLE}"'

remark_change_fqn, planned_stop_fqn, final_amt_fqn, ideal_fqn


('"i_daily_report"."j_remark_change_day_daily"',
 '"i_daily_report"."i_planned_stop_time_day_daily"',
 '"i_daily_report"."a_station_day_daily_final_amount"',
 '"e1_FCT_ct"."fct_whole_op_ct"')

In [5]:
# =========================
# 5) 스키마 탐지(컬럼 존재 여부) - planned stop / ideal table이 환경에 따라 다를 수 있어 방어적으로 처리
# =========================
def get_columns(engine: Engine, schema: str, table: str) -> List[str]:
    sql = text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :schema AND table_name = :table
        ORDER BY ordinal_position
    """)
    with engine.begin() as conn:
        rows = conn.execute(sql, {"schema": schema, "table": table}).fetchall()
    return [r[0] for r in rows]

cols_remark = get_columns(engine, SAVE_SCHEMA, remark_change_table)
cols_stop   = get_columns(engine, SAVE_SCHEMA, planned_stop_table)
cols_final  = get_columns(engine, SAVE_SCHEMA, final_amt_table)
cols_ideal  = get_columns(engine, IDEAL_SCHEMA, IDEAL_TABLE)

cols_remark, cols_stop, cols_final[:20], cols_ideal[:20]


(['prod_day',
  'station',
  'shift_type',
  'at_time',
  'from_remark',
  'to_remark',
  'updated_at'],
 ['prod_day',
  'shift_type',
  'from_time',
  'to_time',
  'Total 계획 정지 시간',
  'total_planned_time',
  'updated_at'],
 ['prod_day',
  'shift_type',
  'station',
  'pn',
  'remark',
  'A시간대(08:30:00 ~ 10:29:59)',
  'B시간대(10:30:00 ~ 12:29:59)',
  'C시간대(12:30:00 ~ 14:29:59)',
  'D시간대(14:30:00 ~ 16:29:59)',
  'E시간대(16:30:00 ~ 18:29:59)',
  'F시간대(18:30:00 ~ 20:29:59)',
  '합계',
  'updated_at'],
 ['id',
  'station',
  'remark',
  'month',
  'ct_eq',
  'uph',
  'final_ct',
  'created_at',
  'updated_at'])

In [6]:
# =========================
# 6) 데이터 로드
# =========================
def load_remark_changes() -> pd.DataFrame:
    # expected cols: prod_day, station, shift_type, at_time, from_remark, to_remark
    sql = text(f"""
        SELECT *
        FROM {remark_change_fqn}
        WHERE prod_day = :prod_day
          AND shift_type = :shift_type
          AND station = ANY(:stations)
        ORDER BY station, at_time
    """)
    with engine.begin() as conn:
        df = pd.read_sql(sql, conn, params={"prod_day": PROD_DAY, "shift_type": SHIFT_TYPE, "stations": STATIONS})
    return df

def load_planned_stops() -> pd.DataFrame:
    # from_time/to_time 필수. (station 컬럼이 있으면 station별)
    sql = text(f"""
        SELECT *
        FROM {planned_stop_fqn}
        WHERE prod_day = :prod_day
          AND shift_type = :shift_type
    """)
    with engine.begin() as conn:
        df = pd.read_sql(sql, conn, params={"prod_day": PROD_DAY, "shift_type": SHIFT_TYPE})
    return df

def load_final_amounts() -> pd.DataFrame:
    sql = text(f"""
        SELECT *
        FROM {final_amt_fqn}
        WHERE prod_day = :prod_day
          AND shift_type = :shift_type
          AND station = ANY(:stations)
    """)
    with engine.begin() as conn:
        df = pd.read_sql(sql, conn, params={"prod_day": PROD_DAY, "shift_type": SHIFT_TYPE, "stations": STATIONS})
    return df

df_remark = load_remark_changes()
df_stop   = load_planned_stops()
df_final  = load_final_amounts()

df_remark.head(), df_stop.head(), df_final.head()


(   prod_day  station shift_type   at_time from_remark to_remark  \
 0  20260128  Vision1        day  10:56:20      Non-PD        PD   
 1  20260128  Vision2        day  10:58:21      Non-PD        PD   
 
                         updated_at  
 0 2026-02-02 03:15:26.816824+00:00  
 1 2026-02-02 03:15:26.816824+00:00  ,
    prod_day shift_type from_time   to_time Total 계획 정지 시간  total_planned_time  \
 0  20260128        day                                10분                 600   
 1  20260128        day  10:50:00  11:00:00            10분                 600   
 
                  updated_at  
 0 2026-02-02 09:19:08+00:00  
 1 2026-02-02 09:19:08+00:00  ,
    prod_day shift_type  station        pn  remark A시간대(08:30:00 ~ 10:29:59)  \
 0  20260128        day  Vision1  35643009  Non-PD        PASS: 339, FAIL: 0   
 1  20260128        day  Vision1  35930927      PD                      None   
 2  20260128        day  Vision2  35643009  Non-PD        PASS: 374, FAIL: 0   
 3  20260128     

In [7]:
# =========================
# 7) PASS 수량 파싱 (예: 'PASS:508, FAIL:1' -> 508)
# =========================
PASS_RE = re.compile(r"PASS\s*[:=]\s*(\d+)", re.IGNORECASE)

def parse_pass(x) -> int:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return 0
    s = str(x)
    m = PASS_RE.search(s)
    return int(m.group(1)) if m else 0

# final amount 테이블에서 합계 컬럼 후보 자동 탐지
# 우선순위: '합계' -> 'total' -> 'sum' -> 첫번째로 PASS 문자열이 발견되는 컬럼
def detect_total_col(df: pd.DataFrame) -> str:
    candidates = []
    for c in df.columns:
        if c == "합계":
            return c
        if c.lower() in ("total", "sum", "total_amount", "overall", "grand_total"):
            candidates.append(c)
    # 후보 중 PASS 패턴이 실제로 존재하는 컬럼 우선
    for c in candidates:
        if df[c].astype(str).str.contains("PASS", case=False, na=False).any():
            return c
    # 전수 탐색
    for c in df.columns:
        if df[c].astype(str).str.contains("PASS", case=False, na=False).any():
            return c
    raise ValueError("Could not detect total column containing PASS/FAIL string. Please set manually.")

TOTAL_COL = detect_total_col(df_final) if not df_final.empty else "합계"
TOTAL_COL


'합계'

In [8]:
# =========================
# 8) remark 기본값(변경 이벤트 없을 때) 확보
#    규칙: final_amount 테이블 remark 컬럼에 존재하는 단 하나의 값 사용
# =========================
def get_base_remark_for_station(df_final: pd.DataFrame, station: str) -> Optional[str]:
    s = df_final[df_final["station"] == station]
    if s.empty:
        return None
    vals = sorted(set(str(v) for v in s["remark"].dropna().unique()))
    if len(vals) == 0:
        return None
    if len(vals) > 1:
        # 명세상 1개여야 하지만 방어적으로 첫 값 사용 + 경고
        print(f"[WARN] station={station} has multiple remark values in final_amount: {vals} -> using first")
    return vals[0]

base_remark = {st: get_base_remark_for_station(df_final, st) for st in STATIONS}
base_remark


[WARN] station=Vision1 has multiple remark values in final_amount: ['Non-PD', 'PD'] -> using first
[WARN] station=Vision2 has multiple remark values in final_amount: ['Non-PD', 'PD'] -> using first


{'Vision1': 'Non-PD', 'Vision2': 'Non-PD'}

In [20]:
# =========================
# 9) station별 remark 구간 생성
# =========================
def build_remark_segments_for_station(df_remark: pd.DataFrame, station: str) -> List[tuple[str, Interval]]:
    """
    return: list[(remark, Interval)] covering the whole shift window (end 포함, duration=end-start)
    규칙:
      - at_time은 '이전 remark의 마지막 시각(포함)'
      - 다음 remark는 at_time+1초부터
    """
    events = df_remark[df_remark["station"] == station].copy()
    events = events.sort_values("at_time")
    segs: List[tuple[str, Interval]] = []
    cur_start = WINDOW_START
    if events.empty:
        r0 = base_remark.get(station)
        if r0 is None:
            raise ValueError(f"No remark change events AND no base remark in final_amount for station={station}")
        segs.append((r0, Interval(WINDOW_START, WINDOW_END)))
        return segs

    # 첫 구간 remark는 첫 이벤트의 from_remark
    cur_remark = str(events.iloc[0]["from_remark"])
    for _, row in events.iterrows():
        at_time = str(row["at_time"])
        at_dt = datetime.combine(WINDOW_START.date(), parse_hms(at_time), tzinfo=KST)
        # night인 경우, at_time이 자정 이후라면 날짜를 +1로 이동
        if SHIFT_TYPE == "night" and at_dt < WINDOW_START:
            at_dt = at_dt + timedelta(days=1)

        # 구간: cur_start ~ at_dt (end 포함 semantic, duration=end-start)
        segs.append((cur_remark, Interval(cur_start, at_dt)))

        # 다음 구간 시작 = at_dt + 1초
        cur_start = at_dt + timedelta(seconds=1)
        cur_remark = str(row["to_remark"])

    # 마지막 구간
    if cur_start < WINDOW_END:
        segs.append((cur_remark, Interval(cur_start, WINDOW_END)))
    elif cur_start == WINDOW_END:
        # 남는 시간이 0초라서 무시 가능
        pass

    return segs

segments = {st: build_remark_segments_for_station(df_remark, st) for st in STATIONS}
segments

{'Vision1': [('Non-PD',
   Interval(start=datetime.datetime(2026, 1, 28, 8, 30, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')), end=datetime.datetime(2026, 1, 28, 10, 56, 20, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')))),
  ('PD',
   Interval(start=datetime.datetime(2026, 1, 28, 10, 56, 21, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')), end=datetime.datetime(2026, 1, 28, 20, 29, 59, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul'))))],
 'Vision2': [('Non-PD',
   Interval(start=datetime.datetime(2026, 1, 28, 8, 30, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')), end=datetime.datetime(2026, 1, 28, 10, 58, 21, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')))),
  ('PD',
   Interval(start=datetime.datetime(2026, 1, 28, 10, 58, 22, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')), end=datetime.datetime(2026, 1, 28, 20, 29, 59, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul'))))]}

In [18]:
# =========================
# 10) planned stop 구간 로드/합집합  (FIXED)
#     - from_time/to_time 비어있거나 0초(=실구간 없음)인 row 스킵
#     - 시간 포맷 유연 파싱 지원
# =========================
import math
import re
from datetime import time as dtime

def _is_missing_time(v) -> bool:
    """DB에서 들어오는 from_time/to_time이 비어있는 케이스 방어."""
    if v is None:
        return True
    # pandas NaN
    if isinstance(v, float) and math.isnan(v):
        return True
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return True
    # '0', '0초' 같은 케이스 방어 (시간으로 해석 불가)
    if s in ("0", "0초", "00", "0000", "000000"):
        return True
    return False

def _parse_time_flexible(v) -> dtime:
    """
    planned stop from/to 시간 파싱(유연).
    허용:
      - HH:MI:SS
      - HH:MI
      - HH:MI:SS.xx (소수초 제거)
      - HHMMSS / HMMSS / HHMM
      - time 객체 (psycopg2가 time으로 주는 경우)
    """
    if isinstance(v, dtime):
        return v

    s = str(v).strip()
    # 소수초 제거: "20:23:25.80" -> "20:23:25", "104000.00" -> "104000"
    if "." in s:
        s = s.split(".", 1)[0].strip()

    if ":" in s:
        parts = s.split(":")
        if len(parts) == 2:
            hh, mm = parts
            ss = "0"
        else:
            hh, mm, ss = parts[0], parts[1], parts[2]
        return dtime(int(hh), int(mm), int(ss))

    # 콜론 없으면 숫자만 추출 (HHMMSS / HMMSS / HHMM)
    digits = re.sub(r"[^0-9]", "", s)
    if digits == "":
        raise ValueError(f"invalid time: {v}")

    if len(digits) == 5:   # HMMSS -> 0HMMSS
        digits = "0" + digits

    if len(digits) == 6:   # HHMMSS
        hh = int(digits[0:2]); mm = int(digits[2:4]); ss = int(digits[4:6])
        return dtime(hh, mm, ss)

    if len(digits) == 4:   # HHMM
        hh = int(digits[0:2]); mm = int(digits[2:4])
        return dtime(hh, mm, 0)

    raise ValueError(f"unsupported time format: {v} (digits={digits})")

def stops_to_intervals(df_stop: pd.DataFrame, station: Optional[str]) -> List[Interval]:
    d = df_stop.copy()

    # station 컬럼이 있으면 station별
    if station is not None and "station" in d.columns:
        d = d[d["station"] == station]

    ivs: List[Interval] = []

    for _, row in d.iterrows():
        ft = row.get("from_time", None)
        tt = row.get("to_time", None)

        # ✅ from/to가 비어있으면 실구간 없음 → 스킵
        if _is_missing_time(ft) or _is_missing_time(tt):
            # 단, 혹시라도 from/to는 비었지만 total_planned_time이 >0 인 이상 케이스라면
            # (정책이 필요하므로) 일단 스킵 + 경고만 찍음.
            tpt = row.get("total_planned_time", None)
            try:
                tpt_val = float(tpt) if tpt is not None and not (isinstance(tpt, float) and math.isnan(tpt)) else 0.0
            except Exception:
                tpt_val = 0.0
            if tpt_val > 0:
                print(f"[WARN] planned_stop row has total_planned_time>0 but missing from/to -> skipped. row={dict(row)}")
            continue

        # 시간 파싱
        ft_t = _parse_time_flexible(ft)
        tt_t = _parse_time_flexible(tt)

        sdt = datetime.combine(WINDOW_START.date(), ft_t, tzinfo=KST)
        edt = datetime.combine(WINDOW_START.date(), tt_t, tzinfo=KST)

        # night인 경우 자정 이후 시간대는 날짜 +1로 이동
        if SHIFT_TYPE == "night":
            if sdt < WINDOW_START:
                sdt += timedelta(days=1)
            if edt < WINDOW_START:
                edt += timedelta(days=1)

        # 방어: 뒤집힌 구간이면 swap
        if edt < sdt:
            sdt, edt = edt, sdt

        # 길이 0(또는 음수)면 스킵
        if edt <= sdt:
            continue

        ivs.append(Interval(sdt, edt))

    return merge_intervals(ivs)

# station별 planned stop 합집합
stop_union: Dict[str, List[Interval]] = {}

if "station" in df_stop.columns:
    for st in STATIONS:
        stop_union[st] = stops_to_intervals(df_stop, st)
else:
    common = stops_to_intervals(df_stop, None)
    for st in STATIONS:
        stop_union[st] = common

# (선택) 확인용: station별 합집합 총 초
{st: total_seconds(ivs) for st, ivs in stop_union.items()}

[WARN] planned_stop row has total_planned_time>0 but missing from/to -> skipped. row={'prod_day': '20260128', 'shift_type': 'day', 'from_time': '', 'to_time': '', 'Total 계획 정지 시간': '10분', 'total_planned_time': 600, 'updated_at': Timestamp('2026-02-02 09:19:08+0000', tz='UTC')}


{'Vision1': 600, 'Vision2': 600}

In [19]:
# =========================
# 11) remark별 planned stop seconds 계산
#     - 이벤트 없으면 total_planned_time 우선 사용(있을 때), 없으면 union 길이 사용
#     - 이벤트 있으면: (planned_stop_union) ∩ (remark_segment) 를 합산
# =========================
def get_total_planned_time_if_available(df_stop: pd.DataFrame) -> Optional[int]:
    if "total_planned_time" not in df_stop.columns:
        return None
    # 테이블이 shift 단위 1행일 가능성 + station별일 가능성 모두 방어
    s = df_stop.dropna(subset=["total_planned_time"])
    if s.empty:
        return None
    # 숫자/문자 혼재 방어
    try:
        return int(float(s.iloc[0]["total_planned_time"]))
    except Exception:
        return None

total_planned_time_shift = get_total_planned_time_if_available(df_stop)

planned_sec_by_station_remark: Dict[tuple[str,str], int] = {}

for st in STATIONS:
    segs = segments[st]
    has_events = not df_remark[df_remark["station"] == st].empty

    if not has_events:
        r0 = base_remark[st]
        if r0 is None:
            raise ValueError(f"station={st}: base remark missing")
        if total_planned_time_shift is not None:
            planned_sec_by_station_remark[(st, r0)] = int(total_planned_time_shift)
        else:
            planned_sec_by_station_remark[(st, r0)] = int(total_seconds(stop_union[st]))
        # 나머지 remark는 0
        for r in REMARKS:
            planned_sec_by_station_remark.setdefault((st, r), 0)
        continue

    # events 있는 경우: overlap 기반
    for r in REMARKS:
        planned_sec_by_station_remark[(st, r)] = 0

    for r, seg_iv in segs:
        # planned stop union과 seg 겹침
        sec = 0
        for ps in stop_union[st]:
            sec += overlap_seconds(ps, seg_iv)
        planned_sec_by_station_remark[(st, r)] += sec

planned_sec_by_station_remark


{('Vision1', 'PD'): 219,
 ('Vision1', 'Non-PD'): 380,
 ('Vision2', 'PD'): 98,
 ('Vision2', 'Non-PD'): 501}

In [12]:
# =========================
# 12) 실제 생산량(PASS) 계산: station+pn+remark 그룹 -> PASS 합
# =========================
if df_final.empty:
    raise ValueError("final amount table returned 0 rows. Check prod_day/shift_type.")

df_final_work = df_final[["station", "pn", "remark", TOTAL_COL]].copy()
df_final_work["pass_qty"] = df_final_work[TOTAL_COL].apply(parse_pass)

actual_by_station_remark = (
    df_final_work.groupby(["station", "remark"], as_index=False)["pass_qty"].sum()
    .rename(columns={"pass_qty": "actual_pass_qty"})
)

actual_by_station_remark


,station,remark,actual_pass_qty
0,Vision1,Non-PD,418
1,Vision1,PD,1167
2,Vision2,Non-PD,440
3,Vision2,PD,1179


In [13]:
# =========================
# 13) Ideal_ct 로드 (기준일: 20260128, day/night 기준 고정)
#     station: left/right, remark: PD/Non-PD, ct_eq: 최소값
#     mapping: left -> Vision1, right -> Vision2 (일반적으로)
# =========================
SIDE_TO_STATION = {"left": "Vision1", "right": "Vision2"}

# ideal table 컬럼명이 환경마다 다를 수 있어 후보를 둠
def pick_col(cols: List[str], candidates: List[str], required=True) -> Optional[str]:
    for c in candidates:
        if c in cols:
            return c
    if required:
        raise ValueError(f"Missing columns, tried: {candidates}, have: {cols}")
    return None

ideal_col_station = pick_col(cols_ideal, ["station"])
ideal_col_remark  = pick_col(cols_ideal, ["remark"])
ideal_col_ct      = pick_col(cols_ideal, ["ct_eq", "final_ct", "ct"])
ideal_col_prod    = pick_col(cols_ideal, ["prod_day", "end_day", "day", "work_day"], required=False)
ideal_col_shift   = pick_col(cols_ideal, ["shift_type", "shift"], required=False)

# 기준일/shift 필터 구성(컬럼 존재할 때만)
where = []
params = {}

if ideal_col_prod is not None:
    where.append(f"{ideal_col_prod} = :base_prod_day")
    params["base_prod_day"] = IDEAL_BASE_PROD_DAY
if ideal_col_shift is not None:
    where.append(f"{ideal_col_shift} = :shift_type")
    params["shift_type"] = SHIFT_TYPE

where_sql = ("WHERE " + " AND ".join(where)) if where else ""

sql_ideal = text(f"""
    SELECT {ideal_col_station} AS side,
           {ideal_col_remark}  AS remark,
           MIN({ideal_col_ct}) AS ideal_ct_sec
    FROM {ideal_fqn}
    {where_sql}
    GROUP BY {ideal_col_station}, {ideal_col_remark}
""")

with engine.begin() as conn:
    df_ideal = pd.read_sql(sql_ideal, conn, params=params)

# side -> Vision station 매핑
df_ideal["station"] = df_ideal["side"].astype(str).str.lower().map(SIDE_TO_STATION)

df_ideal


,side,remark,ideal_ct_sec,station
0,whole,Non-PD,NaN,NaN
1,left,Non-PD,15.18,Vision1
2,whole,PD,NaN,NaN
3,right,PD,17.67,Vision2
4,left,PD,18.35,Vision1
5,right,Non-PD,15.44,Vision2


In [16]:
# =========================
# 14) OEE 계산 (43200초/half-open 기준 최종)
#     - shift window: [start, end) 로 12시간 = 43200초
#     - remark segment: boundary = at_time + 1초
#       이전: [cur_start, boundary) / 다음: [boundary, ...)
#     - effective_sec = remark_window_sec - planned_stop_sec (remark별 overlap)
#     - OEE(너가 말하는 지표) = actual_pass_qty / ideal_qty
# =========================

# (안전) WINDOW_SECONDS가 43200인지 확인
if WINDOW_SECONDS != 43200:
    print(f"[WARN] WINDOW_SECONDS={WINDOW_SECONDS} (expected 43200 for this cell)")

def remark_window_seconds_for(station: str, remark: str) -> int:
    """
    segments[station] = [(remark, Interval), ...]
    Interval은 [start, end) (end exclusive)로 해석.
    remark가 유지된 구간 길이 합을 초로 리턴.
    """
    sec = 0
    for r, iv in segments[station]:
        if r == remark:
            # half-open: duration = end - start
            sec += int((iv.end - iv.start).total_seconds())
    return sec

# 디버그: station별 remark 시간 합이 43200인지 체크
_sum_check = {st: sum(remark_window_seconds_for(st, r) for r in REMARKS) for st in STATIONS}
print("[CHECK] remark_window_sec sum by station:", _sum_check)

rows = []
for st in STATIONS:
    for r in REMARKS:
        # 1) remark별 윈도우(해당 remark가 실제로 유지된 시간)
        remark_window_sec = remark_window_seconds_for(st, r)

        # remark가 0초면(그 remark를 안 쓴 shift) 결과행 자체를 생략(원하면 남겨도 됨)
        if remark_window_sec <= 0:
            continue

        # 2) planned stop: 이미 11번 셀에서 (planned stop union) ∩ (remark segment)로 잘 분배된 값
        planned_sec = int(planned_sec_by_station_remark.get((st, r), 0))

        # 방어: planned stop이 remark_window를 넘으면 cap
        if planned_sec > remark_window_sec:
            print(f"[WARN] planned_sec > remark_window_sec (station={st}, remark={r}) "
                  f"{planned_sec} > {remark_window_sec} -> capped")
            planned_sec = remark_window_sec

        # 3) 유효 가동시간(remark별)
        effective_sec = remark_window_sec - planned_sec

        # 4) Ideal CT (20260128 고정 day/night)
        ideal_ct_row = df_ideal[(df_ideal["station"] == st) & (df_ideal["remark"] == r)]
        ideal_ct = float(ideal_ct_row.iloc[0]["ideal_ct_sec"]) if not ideal_ct_row.empty else None

        # 5) 실제 PASS (station+remark)
        actual_row = actual_by_station_remark[
            (actual_by_station_remark["station"] == st) &
            (actual_by_station_remark["remark"] == r)
        ]
        actual = int(actual_row.iloc[0]["actual_pass_qty"]) if not actual_row.empty else 0

        # 6) Ideal 생산량
        ideal_qty = (effective_sec / ideal_ct) if (ideal_ct and ideal_ct > 0) else None

        # 7) 최종 KPI (너가 말하는 OEE) = Actual / Ideal
        oee = (actual / ideal_qty) if (ideal_qty is not None and ideal_qty > 0) else None

        # 참고용: Ideal/Actual (원하면 제거 가능)
        inv_ratio = (ideal_qty / actual) if (ideal_qty is not None and actual > 0) else None

        rows.append({
            "prod_day": PROD_DAY,
            "shift_type": SHIFT_TYPE,
            "station": st,
            "remark": r,
            "remark_window_sec": remark_window_sec,     # ✅ PD/Non-PD 잘린 시간
            "planned_stop_sec": planned_sec,            # ✅ remark별 planned stop overlap
            "effective_sec": effective_sec,             # ✅ remark별 유효가동
            "ideal_ct_sec": ideal_ct,
            "ideal_qty": ideal_qty,
            "actual_pass_qty": actual,
            "oee_actual_over_ideal": oee,               # ✅ 메인 KPI
            "oee_ideal_over_actual": inv_ratio,         # (참고)
            "updated_at": datetime.now(tz=KST),
        })

df_oee = pd.DataFrame(rows).sort_values(["station", "remark"]).reset_index(drop=True)
df_oee

[WARN] WINDOW_SECONDS=43199 (expected 43200 for this cell)
[CHECK] remark_window_sec sum by station: {'Vision1': 43198, 'Vision2': 43198}


,prod_day,shift_type,station,remark,remark_window_sec,planned_stop_sec,effective_sec,ideal_ct_sec,ideal_qty,actual_pass_qty,oee_actual_over_ideal,oee_ideal_over_actual,updated_at
0,20260128,day,Vision1,Non-PD,8780,380,8400,15.18,553.359684,418,0.755386,1.323827,2026-02-02 18:46:42.902598+09:00
1,20260128,day,Vision1,PD,34418,219,34199,18.35,1863.705722,1167,0.626172,1.597006,2026-02-02 18:46:42.900199+09:00
2,20260128,day,Vision2,Non-PD,8901,501,8400,15.44,544.041451,440,0.808762,1.236458,2026-02-02 18:46:42.906441+09:00
3,20260128,day,Vision2,PD,34297,98,34199,17.67,1935.427278,1179,0.609168,1.641584,2026-02-02 18:46:42.904438+09:00


In [15]:
# =========================
# 15) (선택) 상세 디버깅 출력: station별 remark 구간 + planned stop union 길이
# =========================
def pretty_iv(iv: Interval) -> str:
    return f"{iv.start.strftime('%Y-%m-%d %H:%M:%S')} ~ {iv.end.strftime('%Y-%m-%d %H:%M:%S')} ({int((iv.end-iv.start).total_seconds())}s)"

for st in STATIONS:
    print(f"\n=== {st} ===")
    print("[remark segments]")
    for r, iv in segments[st]:
        print(f"  - {r}: {pretty_iv(iv)}")
    print("[planned stop union]")
    for iv in stop_union[st]:
        print(f"  - {pretty_iv(iv)}")
    print("[planned_stop_sec by remark]")
    for r in REMARKS:
        print(f"  - {r}: {planned_sec_by_station_remark.get((st,r),0)}")



=== Vision1 ===
[remark segments]
  - Non-PD: 2026-01-28 08:30:00 ~ 2026-01-28 10:56:20 (8780s)
  - PD: 2026-01-28 10:56:21 ~ 2026-01-28 20:29:59 (34418s)
[planned stop union]
  - 2026-01-28 10:50:00 ~ 2026-01-28 11:00:00 (600s)
[planned_stop_sec by remark]
  - PD: 219
  - Non-PD: 380

=== Vision2 ===
[remark segments]
  - Non-PD: 2026-01-28 08:30:00 ~ 2026-01-28 10:58:21 (8901s)
  - PD: 2026-01-28 10:58:22 ~ 2026-01-28 20:29:59 (34297s)
[planned stop union]
  - 2026-01-28 10:50:00 ~ 2026-01-28 11:00:00 (600s)
[planned_stop_sec by remark]
  - PD: 98
  - Non-PD: 501


## (선택) 결과를 DB에 저장하고 싶다면
백엔드-11 설계서에는 **저장 대상 테이블**이 명시되어 있지 않아서, 여기서는 계산/표시까지만 합니다.

원하는 테이블명(예: `i_daily_report.k_oee_day_daily` / `k_oee_night_daily` 등)을 정하면:
- `CREATE TABLE ...`
- `UPSERT (prod_day, shift_type, station, remark)` 키 기준
형태로 바로 추가해드릴 수 있습니다.
